In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xarray as xr

# import xarray as xr
# import numpy as np
from scipy.interpolate import griddata

In [2]:
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
import matplotlib.pyplot as plt # Optional: for visualization

In [4]:
!ncdump -h /discover/nobackup/projects/gmao/merra21c/TSE_staging/e5303_m21c_jan18/archive/obs/Y2022/M01/D01/H00/e5303_m21c_jan18.diag_conv_t_ges.20220101_00z.nc4 

netcdf e5303_m21c_jan18.diag_conv_t_ges.20220101_00z {
dimensions:
	nobs = UNLIMITED ; // (137365 currently)
	Station_ID_maxstrlen = 8 ;
	Observation_Class_maxstrlen = 7 ;
	Bias_Correction_Terms_arr_dim = 3 ;
variables:
	char Station_ID(nobs, Station_ID_maxstrlen) ;
	char Observation_Class(nobs, Observation_Class_maxstrlen) ;
	int Observation_Type(nobs) ;
	int Observation_Subtype(nobs) ;
	float Latitude(nobs) ;
	float Longitude(nobs) ;
	float Station_Elevation(nobs) ;
	float Pressure(nobs) ;
	float Height(nobs) ;
	float Time(nobs) ;
	float LaunchTime(nobs) ;
	float Prep_QC_Mark(nobs) ;
	float Setup_QC_Mark(nobs) ;
	float Prep_Use_Flag(nobs) ;
	float Analysis_Use_Flag(nobs) ;
	float Nonlinear_QC_Rel_Wgt(nobs) ;
	float Errinv_Input(nobs) ;
	float Errinv_Adjust(nobs) ;
	float Errinv_Final(nobs) ;
	float Error_Input(nobs) ;
	float Error_Adjust(nobs) ;
	float Observation(nobs) ;
	float Obs_Minus_Forecast_adjusted(nobs) ;
	float Obs_Minus_Forecast_unadjusted(nobs) ;
	float Forecast_unadjuste

In [ ]:
!ncdump -h  /discover/nobackup/projects/QEFM/qefm-core/qefm/models/src/FMGraphCast/../../checkpoints/graphcast/source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc

In [6]:
    import xarray as xr
    import numpy as np
    from scipy.interpolate import griddata
    import pandas as pd
    @staticmethod

    def obs_to_gridded_netcdf2(input_file, output_file, var_name, grid_resolution=0.25, method='linear'):
        """
        Converts an observation-based (scattered points) NetCDF file to a gridded NetCDF file.

        Args:
            input_file (str): Path to the input NetCDF file.
            output_file (str): Path to save the output gridded NetCDF file.
            var_name (str): The name of the variable to interpolate (e.g., 'temperature').
            grid_resolution (float): The desired resolution of the new grid in degrees.
            method (str): Interpolation method ('linear', 'nearest', 'cubic').
        """
        print("obs_to_gridded_netcdf2(" + str(input_file) + " " + str(output_file) + 
              " " + str(var_name) + " " + str(grid_resolution) + " " + str(method) + ")")
        # 1. Load the input data using xarray
        ds_obs = xr.open_dataset(input_file)
        
        # Extract coordinates and the variable data
        # Assuming the input NetCDF has 'lat', 'lon', and the target 'var_name' as dimensions or variables
        # The dimensions in the observation file might be 'site_id' or 'point'
        try:
            lats_obs = ds_obs['Latitude'].values
            lons_obs = ds_obs['Longitude'].values
            level_values_obs = ds_obs['Pressure'].values
            # level_values_obs = (ds_obs['Pressure'].values).astype('int16')
            # print(level_values_obs, level_values_obs.flatten())
            # data_int = ds_obs['Pressure'].astype('int16')
            # print(data_int)
            values_obs = ds_obs[var_name].values
        except KeyError as e:
            print(f"Error: Missing expected coordinate or variable name: {e}")
            return

        # Prepare input points for scipy.interpolate.griddata
        # Flatten coordinates and values if they are not already 1D
        points = np.column_stack((lons_obs.flatten(), lats_obs.flatten()))
        values = values_obs.flatten()

        # 2. Define the target regular grid
        # For now, hard-code min/max from GraphCast init state (GI)
        lon_min, lon_max = 0.0, 359.75
        lat_min, lat_max = -90.0, 90.0
        # Determine the extent of the original data to create a reasonable grid
        # lon_min, lon_max = lons_obs.min(), lons_obs.max()
        # lat_min, lat_max = lats_obs.min(), lats_obs.max()

        # Create the new regular grid coordinates
        new_lons = np.arange(lon_min, lon_max + grid_resolution, grid_resolution)
        new_lats = np.arange(lat_min, lat_max + grid_resolution, grid_resolution)
        X_new, Y_new = np.meshgrid(new_lons, new_lats)

        # 3. Perform the interpolation using scipy.interpolate.griddata
        print(f"Starting interpolation using method: '{method}'...")
        gridded_values = griddata(points, values, (X_new, Y_new), method=method)
        print("Interpolation complete.")

        # constant = 1000000
        # numpy_gridded_values = numpy_gridded_values * constant
        numpy_gridded_values = np.array(gridded_values)
        gridded_values = numpy_gridded_values.tolist()

        # 4. Create a new xarray Dataset for the gridded data
        # Add coordinates as 1D arrays for a standard gridded NetCDF format
        # ds_gridded = xr.Dataset(
        #     {
        #         var_name: (("lat", "lon"), gridded_values),
        #         'Pressure': (("lat", "lon"), level_values_obs),
        #     },
        #     coords={
        #         "lon": new_lons,
        #         "lat": new_lats,
        #         "level": level_values_obs,
        #     }
        # )
        ds_gridded = xr.Dataset(
            {
                var_name: (("lat", "lon"), gridded_values),
            },
            coords={
                "lon": new_lons,
                "lat": new_lats,
                "level": level_values_obs,
            }
        )

        # Copy attributes from the original variable to the new one (optional but recommended for CF conventions)
        ds_gridded[var_name].attrs = ds_obs[var_name].attrs
        ds_gridded.attrs = ds_obs.attrs
        
        # 5. Save the new dataset to a NetCDF file
        ds_gridded.to_netcdf(output_file)
        print(f"Successfully saved gridded data to {output_file}")
        return ds_gridded


In [7]:
# Convert observation point data to gridded data using scipy.interpolate 
ds_OB_gridded = obs_to_gridded_netcdf2(
    "/discover/nobackup/projects/QEFM/qefm-core/data/NSE/input/e5303_m21c_jan18.diag_conv_t_ges.20210101_00z.nc4",
    "/discover/nobackup/projects/QEFM/qefm-core/data/NSE/output/e5303_m21c_jan18.diag_conv_t_ges.20210101_00z_gridded_levels.nc4",
    "Observation"
)

obs_to_gridded_netcdf2(/discover/nobackup/projects/QEFM/qefm-core/data/NSE/input/e5303_m21c_jan18.diag_conv_t_ges.20210101_00z.nc4 /discover/nobackup/projects/QEFM/qefm-core/data/NSE/output/e5303_m21c_jan18.diag_conv_t_ges.20210101_00z_gridded_levels.nc4 Observation 0.25 linear)
Starting interpolation using method: 'linear'...
Interpolation complete.
Successfully saved gridded data to /discover/nobackup/projects/QEFM/qefm-core/data/NSE/output/e5303_m21c_jan18.diag_conv_t_ges.20210101_00z_gridded_levels.nc4


In [8]:
ds_grid=ds_OB_gridded
ds_grid

<xarray.Dataset> Size: 9MB
Dimensions:      (lat: 721, lon: 1440, level: 112633)
Coordinates:
  * lon          (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * lat          (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * level        (level) float32 451kB 6.78e+04 3.28e+04 ... 1.008e+05 1.007e+05
Data variables:
    Observation  (lat, lon) float64 8MB 243.4 243.4 243.4 243.4 ... nan nan nan
Attributes:
    Number_of_Predictors:  3
    date_time:             2021010100
    Number_of_state_vars:  867

In [9]:
ds = ds_grid

In [10]:
condition_mask = (ds['level'] >= 840) & (ds['level'] <= 860)
print(condition_mask)
indices = np.where(condition_mask.values)
print(indices[0].size)

<xarray.DataArray 'level' (level: 112633)> Size: 113kB
array([False, False, False, ..., False, False, False])
Coordinates:
  * level    (level) float32 451kB 6.78e+04 3.28e+04 ... 1.008e+05 1.007e+05
42


In [11]:
level_mask = (ds.level >= 840) & (ds.level <= 860)
selected_levels = ds.where(level_mask, drop=True)
#print(selected_levels.size)

# Get corresponding lat/lon coordinates
lats = selected_levels.lat.values
lons = selected_levels.lon.values
print(lats.size,lons.size)
print(lats[0],lons[0])
print(lats[720],lons[1439])
print(ds.level.values)

721 1440
-90.0 0.0
90.0 359.75
[ 67800.  32800.  31800. ... 100540. 100760. 100670.]


In [12]:
obs_avg_temperature_at_level_850 = selected_levels['Observation'].mean()
print(selected_levels, selected_levels['Observation'].size)
print("The magic number for avg. temperature at level=850 [range = 840-860]: ", float(obs_avg_temperature_at_level_850));

<xarray.Dataset> Size: 349MB
Dimensions:      (lat: 721, lon: 1440, level: 42)
Coordinates:
  * lon          (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * lat          (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * level        (level) float32 168B 850.0 840.0 860.0 ... 860.0 860.0 840.0
Data variables:
    Observation  (lat, lon, level) float64 349MB 243.4 243.4 243.4 ... nan nan
Attributes:
    Number_of_Predictors:  3
    date_time:             2021010100
    Number_of_state_vars:  867 43606080
The magic number for avg. temperature at level=850 [range = 840-860]:  275.6564662919043


In [13]:
# 1. Load the input data using xarray
GI_file = "/discover/nobackup/projects/QEFM/qefm-core/data/NSE/input/graphcast/source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc"
ds_GI = xr.open_dataset(GI_file)

# Extract coordinates and the variable data
# Assuming the input NetCDF has 'lat', 'lon', and the target 'var_name' as dimensions or variables
# The dimensions in the observation file might be 'site_id' or 'point'
try:
    batchs_GI = ds_GI['batch'].values
    lats_GI = ds_GI['lat'].values
    lons_GI = ds_GI['lon'].values
    levels_GI = ds_GI['level'].values
    times_GI = ds_GI['time'].values
    temperatures_GI = ds_GI['temperature'].values
except KeyError as e:
    print(f"Error: Missing expected coordinate or variable name: {e}")
print(levels_GI)

[  50  100  150  200  250  300  400  500  600  700  850  925 1000]


In [14]:
# Define the new value you want to assign
level_of_interest = 850 # Example value
for i, (batch, lat, lon, level, time) in enumerate(zip(batchs_GI, lats_GI, lons_GI, levels_GI, times_GI)):
    ds_GI['temperature'].loc[dict(batch=batch, lat=lat, lon=lon, level=level_of_interest, time=time)] = obs_avg_temperature_at_level_850
#    temperatures_GI.loc[dict(batch=batch, lat=lat, lon=lon, level=850, time=time)] = obs_avg_temperature_at_level_850

In [20]:
import os
out_dir = "/discover/nobackup/projects/QEFM/qefm-core/data/NSE/output/graphcast"
#out_file_value = f"graphcast-prediction-{input_source}_date-{date_str}_res-{res}_levels-{levs}_freq-{cfreq}h_steps-{eval_steps}.nc"
out_file_value = "_source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc"
out_file = os.path.join(out_dir, out_file_value)
ds_GI.to_netcdf(out_file)
# days=(6*eval_steps)/24
print("Saved obs-updated temps for lev=", str(level_of_interest), " : \n", out_file, "\n")

Saved obs-updated temps for lev= 850  : 
 /discover/nobackup/projects/QEFM/qefm-core/data/NSE/output/graphcast/_source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc 



In [21]:
!/usr/local/other/nco/5.1.7/bin/ncks -d level,10 -v temperature /discover/nobackup/projects/QEFM/qefm-core/qefm/models/src/FMGraphCast/../../checkpoints/graphcast/source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc output.nc


ncks: output.nc exists---`e'xit, `o'verwrite (i.e., clobber existing file), or `a'ppend (i.e., replace duplicate variables in, and add metadata and new variables to, existing file) (e/o/a)? ^C
